# Stochastic optimization theory

Stochastic optimization accepts noisy gradients because unbiased noise averages into reliable motion.

Full gradients become minibatch estimates sampled from data. Probability supplies unbiasedness, variance, and convergence in expectation. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 271828
rng = np.random.default_rng(SEED)


def sigmoid(z):
    clipped = np.clip(z, -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-clipped))


def soft_threshold(v, threshold):
    return np.sign(v) * np.maximum(np.abs(v) - threshold, 0.0)


def quadratic_loss(A, b, x):
    return 0.5 * float(x @ A @ x) - float(b @ x)


def quadratic_grad(A, b, x):
    return A @ x - b


def rosenbrock_loss(x):
    a = 1.0 - x[0]
    b = x[1] - x[0] ** 2
    ripple = 0.08 * np.sin(3.0 * x[0]) * np.cos(2.0 * x[1])
    return a ** 2 + 35.0 * b ** 2 + ripple


def rosenbrock_grad(x):
    dx = -2.0 * (1.0 - x[0]) - 140.0 * x[0] * (x[1] - x[0] ** 2)
    dy = 70.0 * (x[1] - x[0] ** 2)
    dx = dx + 0.24 * np.cos(3.0 * x[0]) * np.cos(2.0 * x[1])
    dy = dy - 0.16 * np.sin(3.0 * x[0]) * np.sin(2.0 * x[1])
    return np.array([dx, dy])


def make_logistic_data(n=96, d=2, seed=11):
    local = np.random.default_rng(seed)
    half = n // 2
    pos = local.normal(loc=1.15, scale=0.55, size=(half, d))
    neg = local.normal(loc=-1.05, scale=0.65, size=(n - half, d))
    X = np.vstack([pos, neg])
    y = np.hstack([np.ones(half), -np.ones(n - half)])
    return X, y


def make_sparse_logistic_data(n=140, d=32, seed=23):
    local = np.random.default_rng(seed)
    X = local.normal(size=(n, d))
    mask = local.random(size=X.shape) < 0.72
    X[mask] = 0.0
    true_w = np.zeros(d)
    true_w[:5] = np.array([1.4, -1.1, 0.9, -0.7, 0.45])
    logits = X @ true_w + 0.15 * local.normal(size=n)
    y = np.where(logits >= 0.0, 1.0, -1.0)
    return X, y


def logistic_loss(w, X, y, lam=0.0):
    margins = y * (X @ w)
    data = np.logaddexp(0.0, -margins).mean()
    penalty = 0.5 * lam * float(w @ w)
    return float(data + penalty)


def logistic_grad(w, X, y, lam=0.0):
    margins = y * (X @ w)
    weights = -y * sigmoid(-margins)
    grad = X.T @ weights / X.shape[0]
    return grad + lam * w


def l1_logistic_objective(w, X, y, lam=0.04):
    return logistic_loss(w, X, y, 0.0) + lam * float(np.abs(w).sum())


def project_box(x, lo=-2.0, hi=2.0):
    return np.clip(x, lo, hi)


def project_l2_ball(x, radius=2.0):
    norm = np.linalg.norm(x)
    if norm <= radius:
        return x.copy()
    return x * (radius / norm)


def make_loss_surface_ladder():
    X2, y2 = make_logistic_data()
    Xh, yh = make_sparse_logistic_data()
    d = Xh.shape[1]
    A1 = np.array([[4.0, 1.0], [1.0, 3.0]])
    b1 = np.array([1.0, 2.0])
    A2 = np.array([[45.0, 18.0], [18.0, 9.0]])
    b2 = np.array([1.0, 0.4])
    return [
        {
            "id": "D1",
            "name": "quadratic bowl",
            "x0": np.array([0.0, 0.0]),
            "loss": lambda x, A=A1, b=b1: quadratic_loss(A, b, x),
            "grad": lambda x, A=A1, b=b1: quadratic_grad(A, b, x),
            "project": lambda x: x,
            "dim": 2,
            "info": "2-D SPD quadratic with closed-form minimizer",
        },
        {
            "id": "D2",
            "name": "ill-conditioned quadratic",
            "x0": np.array([1.8, -1.5]),
            "loss": lambda x, A=A2, b=b2: quadratic_loss(A, b, x),
            "grad": lambda x, A=A2, b=b2: quadratic_grad(A, b, x),
            "project": lambda x: x,
            "dim": 2,
            "info": "anisotropic bowl with coupled coordinates",
        },
        {
            "id": "D3",
            "name": "nonconvex Rosenbrock-ripple",
            "x0": np.array([-1.2, 1.0]),
            "loss": rosenbrock_loss,
            "grad": rosenbrock_grad,
            "project": lambda x: x,
            "dim": 2,
            "info": "curved valley plus small multimodal ripple",
        },
        {
            "id": "D4",
            "name": "real logistic loss",
            "x0": np.zeros(2),
            "loss": lambda w, X=X2, y=y2: logistic_loss(w, X, y, 0.02),
            "grad": lambda w, X=X2, y=y2: logistic_grad(w, X, y, 0.02),
            "project": lambda x: x,
            "dim": 2,
            "X": X2,
            "y": y2,
            "info": "NumPy logistic regression objective on a fixed small dataset",
        },
        {
            "id": "D5",
            "name": "high-dimensional sparse constrained case",
            "x0": np.zeros(d),
            "loss": lambda w, X=Xh, y=yh: l1_logistic_objective(w, X, y, 0.04),
            "grad": lambda w, X=Xh, y=yh: logistic_grad(w, X, y, 0.0),
            "project": lambda x: project_l2_ball(x, 2.5),
            "dim": d,
            "X": Xh,
            "y": yh,
            "A": A5,
            "b": b5,
            "info": "32-D sparse logistic objective with L1 and norm constraint",
        },
    ]


def preview_ladder(ladder):
    for rung in ladder:
        sample = rung["x0"][: min(5, rung["dim"])]
        print(rung["id"], rung["name"], "dim=", rung["dim"], "sample=", np.round(sample, 3), "--", rung["info"])


def contour_values(rung, grid=80, span=2.4):
    xs = np.linspace(-span, span, grid)
    ys = np.linspace(-span, span, grid)
    Z = np.zeros((grid, grid))
    base = rung["x0"].astype(float).copy()
    for i, xval in enumerate(xs):
        for j, yval in enumerate(ys):
            probe = base.copy()
            probe[0] = xval
            probe[1] = yval
            Z[j, i] = rung["loss"](probe)
    return xs, ys, Z


def plot_trajectory_summary(results, metric_label="final loss"):
    fig, axes = plt.subplots(2, 5, figsize=(18, 6))
    for col, item in enumerate(results):
        rung = item["rung"]
        path = item["path"]
        losses = item["losses"]
        xs, ys, Z = contour_values(rung)
        axes[0, col].contour(xs, ys, Z, levels=18, cmap="viridis")
        axes[0, col].plot(path[:, 0], path[:, 1], marker="o", markersize=2, linewidth=1)
        axes[0, col].set_title(rung["id"])
        axes[1, col].plot(losses)
        axes[1, col].set_title(metric_label)
        axes[1, col].set_xlabel("iteration")
    plt.tight_layout()


def print_metric_table(results, metric_label="final loss"):
    print(f"{'rung':<4} {'name':<38} {'iters':>6} {metric_label:>14}")
    for item in results:
        final_value = item["metric"]
        iters = len(item["losses"]) - 1
        print(f"{item['rung']['id']:<4} {item['rung']['name']:<38} {iters:>6} {final_value:>14.6f}")

## The concept, built once (D1)

For $F(w)=\frac1n\sum_i\ell_i(w)$, uniform minibatches satisfy $\mathbb{E}[g_B]=\nabla F(w)$.

Check the exact minibatch arithmetic from the lesson. Individual batches wobble, but their sampling target is the full gradient.

In [ ]:
def minibatch_mean(values, indices):
    return float(np.mean(values[indices]))


grads = np.array([-4.0, -16.0, -36.0, -64.0])
full_gradient = grads.mean()
batch_12 = minibatch_mean(grads, np.array([0, 1]))
batch_34 = minibatch_mean(grads, np.array([2, 3]))
variance_b1 = 9.0 / 1.0
variance_b9 = 9.0 / 9.0
variance_b100 = 9.0 / 100.0

assert np.isclose(full_gradient, -30.0)
assert np.isclose(batch_12, -10.0)
assert np.isclose(batch_34, -50.0)
assert np.isclose(variance_b1, 9.0)
assert np.isclose(variance_b9, 1.0)
assert np.isclose(variance_b100, 0.09)
print(full_gradient, batch_12, batch_34)

Build a seeded SGD routine with explicit sampling probabilities. This lets us reproduce unbiased uniform sampling and the biased-sampling pitfall.

In [ ]:
def sgd(rung, steps=140, batch_size=12, eta0=0.18, probabilities=None, seed=SEED):
    local = np.random.default_rng(seed)
    x = rung["x0"].astype(float).copy()
    path = [x.copy()]
    losses = [rung["loss"](x)]
    n = rung.get("X", np.zeros((batch_size, rung["dim"]))).shape[0]
    for t in range(steps):
        eta = eta0 / math.sqrt(t + 1.0)
        if "X" in rung:
            idx = local.choice(n, size=min(batch_size, n), replace=True, p=probabilities)
            Xb = rung["X"][idx]
            yb = rung["y"][idx]
            grad = logistic_grad(x, Xb, yb, 0.0)
        else:
            noise = local.normal(scale=0.08, size=rung["dim"])
            grad = rung["grad"](x) + noise
        x = rung["project"](x - eta * grad)
        path.append(x.copy())
        losses.append(rung["loss"](x))
    return np.array(path), np.array(losses)

## The dataset ladder

Family F4 uses the same inline D1-D5 ladder: quadratic, ill-conditioned, nonconvex, real logistic loss, and high-dimensional sparse constrained loss.

In [ ]:
ladder = make_loss_surface_ladder()
preview_ladder(ladder)

## Run the same method across D1-D5

The metric is the plan's requested final loss, final objective, feasible loss, or primal-dual gap.

In [ ]:
ladder = make_loss_surface_ladder()
results = []
for rung in ladder:
    path, losses = sgd(rung, steps=140, batch_size=10, eta0=0.16, seed=SEED + rung["dim"])
    results.append({"rung": rung, "path": path, "losses": losses, "metric": losses[-1]})
print_metric_table(results, "final loss")

## Results visualization

The closing figure has trajectory-on-contours panels and a summary metric curve.

In [ ]:
plot_trajectory_summary(results, "stochastic loss")
metrics = [item["metric"] for item in results]
plt.figure(figsize=(6, 3))
plt.plot([item["rung"]["id"] for item in results], metrics, marker="o")
plt.ylabel("final loss")
plt.title("SGD final loss by rung")
plt.grid(True, alpha=0.3)
plt.show()

## Pitfall on the hardest rung

Pitfall on D5: calling any minibatch gradient unbiased. Biased sampling changes the target; a large constant step also leaves a noisy cloud.

In [ ]:
lesson_grads = np.array([-4.0, -2.0, 0.0, 2.0])
uniform_mean = lesson_grads.mean()
probabilities = np.array([0.7, 0.1, 0.1, 0.1])
biased_mean = float(probabilities @ lesson_grads)
assert np.isclose(uniform_mean, -1.0)
assert np.isclose(biased_mean, -2.8)

hard = ladder[-1]
n = hard["X"].shape[0]
biased_probs = np.ones(n) * 0.3 / (n - 1)
biased_probs[0] = 0.7
wrong_path, wrong_losses = sgd(hard, steps=80, batch_size=8, eta0=0.16, probabilities=biased_probs, seed=5)
fixed_path, fixed_losses = sgd(hard, steps=80, batch_size=8, eta0=0.16, probabilities=None, seed=5)
print("lesson uniform", uniform_mean, "biased", biased_mean)
print("biased sampling final", wrong_losses[-1])
print("uniform sampling final", fixed_losses[-1])

## Evaluate it + Practice

- Metric: compare the final value to a no-skill baseline that returns the initial point.
- Sanity check: on D1, verify the path moves toward the closed-form quadratic solution or the asserted lesson numbers.
- Ablation: turn off the key idea, such as newest-value updates, the prox, projection, dual lower bound, uniform sampling, or PSD check.
- Failure signals: rising loss, exploding iterates, negative inequality multipliers, invalid lower bounds, or dense non-sparse D5 solutions.
- CPU note: these examples are seeded, small, and NumPy-only; do not execute the notebook as part of rebuilding.

Practice: Increase batch size from 1 to 32 and plot variance of the loss curve.

Practice: Use a constant learning rate and measure the final iterate cloud over five seeds.

Practice: Importance-weight a biased sampler and check whether the mean gradient target is restored.